In [29]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import accuracy_score, classification_report


## 1. Build a small concrete corpus

We use a tiny hand-crafted corpus with three themes:
- politics
- health
- finance

In [30]:
docs = [
    "The government announced a new election campaign and parliament debate.",
    "The minister discussed tax reform and public policy in parliament.",
    "Voters reacted strongly to the new election promises and political speeches.",
    "The opposition criticized the government during the budget debate.",
    "Parliament debated whether new regulations would affect citizens and businesses.",

    "Doctors at the hospital reported progress in vaccine research.",
    "The patient received treatment at the medical center after surgery.",
    "New health policy focuses on prevention, hospitals, and mental health.",
    "Researchers studied disease risk and public health outcomes.",
    "The clinic expanded mental health support and preventive care services.",

    "The bank reported higher profits and stronger market performance.",
    "Investors are worried about inflation, interest rates, and stock prices.",
    "The company announced new earnings and growth in the financial market.",
    "Consumers reduced spending as the economy slowed and prices increased.",
    "The central bank signaled concern about inflation and economic stability."
]

labels = [
    "politics", "politics", "politics", "politics", "politics",
    "health", "health", "health", "health", "health",
    "finance", "finance", "finance", "finance", "finance"
]

df = pd.DataFrame({"text": docs, "label": labels})
df

,text,label
0,The government announced a new election campai...,politics
1,The minister discussed tax reform and public p...,politics
2,Voters reacted strongly to the new election pr...,politics
3,The opposition criticized the government durin...,politics
4,Parliament debated whether new regulations wou...,politics
5,Doctors at the hospital reported progress in v...,health
6,The patient received treatment at the medical ...,health
7,"New health policy focuses on prevention, hospi...",health
8,Researchers studied disease risk and public he...,health
9,The clinic expanded mental health support and ...,health


## 2. CountVectorizer: Bag of Words

`CountVectorizer` builds a document-term matrix of **raw counts**.

Key idea:
- each **row** = one document
- each **column** = one word (or n-gram)
- each value = how many times that word appears in that document

This is the classic **Bag of Words** representation.

In [31]:
count_vec = CountVectorizer(stop_words="english")
X_count = count_vec.fit_transform(df["text"])

print("Sparse matrix shape:", X_count.shape)
print("Number of vocabulary terms:", len(count_vec.get_feature_names_out()))
print("\nFirst 20 terms in the vocabulary:")
print(count_vec.get_feature_names_out()[:20])


Sparse matrix shape: (15, 82)
Number of vocabulary terms: 82

First 20 terms in the vocabulary:
['affect' 'announced' 'bank' 'budget' 'businesses' 'campaign' 'care'
 'center' 'central' 'citizens' 'clinic' 'company' 'concern' 'consumers'
 'criticized' 'debate' 'debated' 'discussed' 'disease' 'doctors']


In [32]:
count_preview = pd.DataFrame(
    X_count[:5, :12].toarray(),
    columns=count_vec.get_feature_names_out()[:12]
)
count_preview

,affect,announced,bank,budget,businesses,campaign,care,center,central,citizens,clinic,company
0,0,1,0,0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,1,0,0,0,0,0,0,0,0
4,1,0,0,0,1,0,0,0,0,1,0,0


## Quick note

For **topic modeling with LDA**, raw counts are usually more natural than TF-IDF.

For **classification and clustering**, TF-IDF is often a stronger baseline.

## 3. TF-IDF: a more informative weighting

`TfidfVectorizer` downweights words that appear in many documents and highlights words that are more distinctive.

Very rough intuition:

- high in this document
- not too common across all documents
- therefore more informative

In [33]:
tfidf_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
X_tfidf = tfidf_vec.fit_transform(df["text"])

print("TF-IDF matrix shape", X_tfidf.shape)
print("Number of features:", len(tfidf_vec.get_feature_names_out()))

tfidf_preview = pd.DataFrame(
    X_tfidf[:5, :12].toarray().round(3),
    columns=tfidf_vec.get_feature_names_out()[:12]
)
tfidf_preview

TF-IDF matrix shape (15, 167)
Number of features: 167


,affect,affect citizens,announced,announced new,bank,bank reported,bank signaled,budget,budget debate,businesses,campaign,campaign parliament
0,0.000,0.000,0.267,0.267,0.0,0.0,0.0,0.000,0.000,0.000,0.308,0.308
1,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000
2,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.000,0.000,0.000,0.000
3,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.343,0.343,0.000,0.000,0.000
4,0.289,0.289,0.000,0.000,0.0,0.0,0.0,0.000,0.000,0.289,0.000,0.000


## 4. Text classification with `make_pipeline()`

Practical workflows:
- vectorize text
- fit classifier
- predict labels

We use:
- `TfidfVectorizer`
- `LogisticRegression`


In [34]:
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], random_state=42, test_size=0.33, stratify=df["label"])

clf_pipe = make_pipeline(TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2),
                         LogisticRegression(max_iter=2000))

clf_pipe.fit(X_train, y_train)
pred = clf_pipe.predict(X_test)

print("Accuracy:", round(accuracy_score(pred, y_test), 3))
print()
print(classification_report(pred, y_test))

Accuracy: 0.6

              precision    recall  f1-score   support

     finance       1.00      0.33      0.50         3
      health       0.50      1.00      0.67         1
    politics       0.50      1.00      0.67         1

    accuracy                           0.60         5
   macro avg       0.67      0.78      0.61         5
weighted avg       0.80      0.60      0.57         5



## 5. Interpret the classifier

In text mining, **interpretation matters**.

For linear models like logistic regression, we can inspect:
- which features are most strongly associated with each class

This is a very useful habit:
> do not stop at the score; inspect what the model learned.

In [35]:
vectorizer = clf_pipe.named_steps["tfidfvectorizer"]
model = clf_pipe.named_steps["logisticregression"]
feature_names = np.array(vectorizer.get_feature_names_out())

for i, class_label in enumerate(model.classes_):
    top = np.argsort(model.coef_[i])[-12:]
    print(f"\nTop features for class '{class_label}':")
    print(feature_names[top])


Top features for class 'finance':
['health' 'public' 'debate' 'government' 'parliament' 'policy' 'new'
 'announced' 'announced new' 'market' 'prices']

Top features for class 'health':
['prices' 'market' 'parliament' 'debate' 'government' 'announced'
 'announced new' 'new' 'policy' 'public' 'health']

Top features for class 'politics':
['health' 'prices' 'market' 'new' 'announced' 'announced new' 'public'
 'policy' 'parliament' 'debate' 'government']


## 6. Document clustering with KMeans

Now we switch to **unsupervised** task

Goal:
- group similar documents without using labels

A classic baseline:

- `TfidfVectorizer`
- `KMeans`

Important:
- **cluster != topic**
- cluster means grouping documents
- topic means a latent theme structure

In [36]:
cluster_vec = TfidfVectorizer(stop_words="english")
X_cluster = cluster_vec.fit_transform(df["text"])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_cluster)

df_clustered = df.copy()
df_clustered["cluster"] = cluster_labels
df_clustered[["label", "cluster", "text"]]

,label,cluster,text
0,politics,2,The government announced a new election campai...
1,politics,2,The minister discussed tax reform and public p...
2,politics,2,Voters reacted strongly to the new election pr...
3,politics,2,The opposition criticized the government durin...
4,politics,2,Parliament debated whether new regulations wou...
5,health,0,Doctors at the hospital reported progress in v...
6,health,0,The patient received treatment at the medical ...
7,health,1,"New health policy focuses on prevention, hospi..."
8,health,1,Researchers studied disease risk and public he...
9,health,1,The clinic expanded mental health support and ...


In [37]:
from sklearn.metrics import adjusted_rand_score

adjusted_rand_score(df["label"], cluster_labels)

0.47896440129449835

In [38]:
terms = np.array(cluster_vec.get_feature_names_out())
centers = kmeans.cluster_centers_

for i in range(3):
    top_idx = centers[i].argsort()[-10:]
    print(f"\nCluster {i} top terms:")
    print(terms[top_idx])


Cluster 0 top terms:
['doctors' 'hospital' 'rates' 'stock' 'investors' 'worried' 'bank'
 'prices' 'reported' 'inflation']

Cluster 1 top terms:
['focuses' 'hospitals' 'prevention' 'outcomes' 'researchers' 'studied'
 'disease' 'risk' 'mental' 'health']

Cluster 2 top terms:
['campaign' 'opposition' 'criticized' 'budget' 'election' 'announced'
 'debate' 'government' 'parliament' 'new']


## 7. Topic modeling with LDA

Topic modeling asks a different question:

> What latent themes organize this corpus?

- `CountVectorizer`
- `LatentDirichletAllocation`

Key ideas:

- each topic = distribution over words